<a href="https://colab.research.google.com/github/johnriskquant/mml-data-science-notes/blob/main/prob-stats/week-1-prob-and-prob-distributions/nvb_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Naive Bayes**


We'll build out a naive bayes algorithm for spam detection to illustrative the usesfulness of the bayes theorem in ML modeling. The Naive Bayes algorithm will leverage Bayes' Theorem in order to determine whether a data point belongs to a specific class.

The algorithm makes a "naive assumption" that each feature is independent of the others. While this assumption almost certainly does not hold true in reality, it makes the algorithm easier to implement and, as you'll see, can lead to impressively useful results.

# **Problem Statement**

Distinguish between spam and non-spam emails given a labeled dataset. For the purpose of this task, spam emails will be labeled as $1$, and non-spam (ham) emails as $0$.

The probability of interest for a given email is denoted as:

$$ P(\text{spam} \mid \text{email}) $$

Recall that:

$$ P(\text{spam} \mid \text{email}) = \frac{P(\text{email} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{email})} $$

Where:

- $ P(\text{spam}) $: Probability of a randomly selected email being spam, equivalent to the proportion of spam emails in the dataset.
- $ P(\text{email} \mid \text{spam}) $: Probability of a specific email occurring given that it is known to be spam.
- $ P(\text{email}) $: Overall probability of the email occurring.

An interesting early "shortcut" you can take in this approach is just ignore the $ P(\text{email}) $ term. The goal of this calculation will be to compare the probability an email is spam to the probability it is ham. Here's the expression for both $ P(\text{spam} \mid \text{email}) $ and $ P(\text{ham} \mid \text{email}) $:

$$ P(\text{spam} \mid \text{email}) = \frac{P(\text{email} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{email})} $$

$$ P(\text{ham} \mid \text{email}) = \frac{P(\text{email} \mid \text{ham}) \cdot P(\text{ham})}{P(\text{email})} $$

Since $ P(\text{email}) > 0 $ and it appears in both expressions, comparing the two probabilities only requires evaluating the numerators and you can ignore this denominator.

# **Required Libraries**

In [2]:
import numpy as np
import pandas as pd
import nltk #nltk: natural lang library
# from nltk.corpus import stopwords
# from nltk import word_tokenize
import string

In [11]:
df_emails = pd.read_csv('/emails.csv')
df_emails.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [12]:
print(f"Number of emails: {len(df_emails)}")
print(f"Proportion of spam emails: {df_emails.spam.sum()/len(df_emails):.4f}")
print(f"Proportion of ham emails: {1-df_emails.spam.sum()/len(df_emails):.4f}")

Number of emails: 5728
Proportion of spam emails: 0.2388
Proportion of ham emails: 0.7612


*Notice the inbalance in the dataset. There are 2x more spam emails than ham emails. This is useful context for our classifier*

# **Preprocessing dataset**


1.   Every email starts with `Subject:` so we remove the text from every email
2.   Randomly shuffle the dataset because all spam emails are the top of the data set. We need shuffled dataset to properly split between train and test datasets



In [13]:
def preprocess_emails(df:pd.DataFrame)->pd.DataFrame:
  """
      Preprocesses email data from a DataFrame.

    Parameters:
    - df (pandas.DataFrame): The input DataFrame containing email data with 'text' and 'spam' columns.

    Returns:
    - tuple: A tuple containing two elements:
        1. X (numpy.array): An array containing email content after removing the "Subject:" prefix.
        2. Y (numpy.array): An array indicating whether each email is spam (1) or ham (0).

    The function shuffles the input DataFrame to avoid biased results in train/test splits.
    It then extracts email content and spam labels, removing the "Subject:" prefix from each email.
  """

  df = df.copy()
  df = df.sample(frac = 1, ignore_index = True, random_state = 42)
  X = df['text'].apply(lambda text: text[9:]).to_numpy()
  Y = df['spam'].to_numpy()
  return X, Y

In [14]:
X, Y = preprocess_emails(df_emails)
print(X[:5])

['re : energy derivatives conference - may 29 , toronto  good morning amy :  vince kaminski will need the following :  an lcd projector to hook up to a lap tap for his presentation  he will have dinner with the conference organizers and speakers on the 29 th .  he will need 2 nights ( the 28 th and the 29 th ) hotel reservations .  he will send you an abstract shortly .  thanks and have a great day !  shirley crenshaw  713 - 853 - 5290  amy aldous on 03 / 31 / 2000 10 : 50 : 11 am  to : shirley . crenshaw @ enron . com  cc :  subject : re : energy derivatives conference - may 29 , toronto  ms . crenshaw ,  thank you for sending the bio so quickly . it \' s exactly what i was looking  for .  we are planning to compile the conference speakers \' papers for distribution  to the participants . while i will not need dr . kaminski \' s contribution for  several weeks , an abstract of his presentation as soon as possible would be  very useful to the conference organizers .  i will also need t

In [15]:
print(Y[:5])

[0 0 0 0 0]


In [16]:
#Is email 42 spam or ham?
email_index = 42
print(f"Email index {email_index}: {X[email_index]}\n\n")
print(f"Class: {Y[email_index]}")

Email index 42: benachrichtung  zum  ( fehlgeschlagen ) ? =  dies ist eine automatisch erstellte benachrichtigung + apw - ber den zustellstatus .  + anw - bermittlung an folgende empf + aoq - nger fehlgeschlagen .  jochenfechtel @ fetra . de


Class: 1


## Preprocess text


*   Stopwords (e.g. pronouns, prepositions) and punctuations are removed as they don't provide any meaningful information for our classifier
*   Split the emails in tokens (tokenization), which are essentially works.



In [17]:
def preprocess_text(X:np.ndarray[str])->np.ndarray[str]:
  """
  Preprocesses a collection of text data by removing stopwords and punctuation.

  Parameters:
  - X (str or array-like): The input text data to be processed. If a single string is provided,
    it will be converted into a one-element numpy array.

  Returns:
  - numpy.array: An array of preprocessed text data, where each element represents a document
    with stopwords and punctuation removed.

  Note:
  - The function uses the Natural Language Toolkit (nltk) library for tokenization and stopword removal.
  - If the input is a single string, it is converted into a one-element numpy array.
  """
  nltk.download('stopwords', quiet=True)
  nltk.download('punkt_tab', quiet=True)
  stop = set(nltk.stem.snowball.stopwords.words('english') + list(string.punctuation))

  #handles a case where single email is passed instead of an array of emails
  if isinstance(X, str):
    X = np.array([X])

  # The result will be stored in a list
  X_preprocessed = []

  for i, email in enumerate(X):
    email = np.array([word.lower() for word in nltk.word_tokenize(email) if word.lower() not in stop]).astype(X.dtype)
    X_preprocessed.append(email)

  if len(X) == 1:
    return X_preprocessed[0]
  return X_preprocessed

In [18]:
# This function may take a few seconds to run. Usually less than 1 minute.
X_treated = preprocess_text(X)
email_index = 989
print(f"Email before preprocessing: {X[email_index]}")
print(f"Email after preprocessing: {X_treated[email_index]}")

Email before preprocessing: marketing for your espeak session  vince :  thanks for your time earlier this week ; i ' m looking forward to your espeak  event .  sarah and i met with our etv contact yesterday , and we will be able to put a  bulleted list on the elevator screens to advertise your espeak . please let  me know what you would like us to post for you , and we will do the rest !  we also have plans to market specifically to the trader community here at  enron , so you should get a high participation rate , especially from those  groups .  thanks , again .  - er
Email after preprocessing: ['marketing' 'espeak' 'session' 'vince' 'thanks' 'time' 'earlier' 'week'
 'looking' 'forward' 'espeak' 'event' 'sarah' 'met' 'etv' 'contact'
 'yesterday' 'able' 'put' 'bulleted' 'list' 'elevator' 'screens'
 'advertise' 'espeak' 'please' 'let' 'know' 'would' 'like' 'us' 'post'
 'rest' 'also' 'plans' 'market' 'specifically' 'trader' 'community'
 'enron' 'get' 'high' 'participation' 'rate' 'espec

## Splitting into train/test
We'll use a 80/20 split here (80% training/ 20% testing)

In [19]:
TRAIN_SIZE = int(0.8*len(X_treated))
X_train = X_treated[:TRAIN_SIZE]
X_test = X_treated[TRAIN_SIZE:]
Y_train = Y[:TRAIN_SIZE]
Y_test =  Y[TRAIN_SIZE:]

In [20]:
print(f"Proportion of spam in train dataset: {sum(Y_train == 1)/len(Y_train):.4f}")
print(f"Proportion of spam in test dataset: {sum(Y_test == 1)/len(Y_test):.4f}")

Proportion of spam in train dataset: 0.2431
Proportion of spam in test dataset: 0.2216


# Implementing the Naive Bayes Algorithm
Remember our goal: Compare $P(\text{spam} \mid \text{email})$ and $P(\text{ham} \mid \text{email})$ to decide which one is greater. It is sufficient to compute only $P(\text{spam}) \cdot P(\text{email} \mid \text{spam})$ and $P(\text{ham}) \cdot P(\text{email} \mid \text{ham})$ to make the comparison.

Note that $P(\text{spam} \mid \text{email})$ is identical to $P(\text{ham} \mid \text{email})$, so let's start on the spam case.

Each email is a list of words. Your goal is to calculate how likely you are to see this list of words, given the email is spam. The way you'll do that is to apply the product rule. Representing an email as $\text{email} = \{\text{word}_1, \text{word}_2, \ldots, \text{word}_n \}$, the computation is:

$$P(\text{email} \mid \text{spam}) = P(\text{word}_1 \mid \text{spam}) \cdot P(\text{word}_2 \mid \text{spam}) \cdots P(\text{word}_n \mid \text{spam})$$

Where:
$$P(\text{word}_1 \mid \text{spam}) = \frac{\text{# spam emails with } \text{word}_1}{\text{# spam emails}}$$

We start by counting occurrence of each word from 1 because if a word never appears (i.e., $P(\text{word} \mid \text{spam}) = 0$), the single word could make the entire probability $0$

In [23]:
def get_word_frequency(X:np.ndarray, Y:np.ndarray)->dict[dict]:
  """
  Calculate the frequency of each word in a set of emails categorized as spam (1) or not spam (0).

  Parameters:
  - X (numpy.array): Array of emails, where each email is represented as a list of words.
  - Y (numpy.array): Array of labels corresponding to each email in X. 1 indicates spam, 0 indicates ham.

  Returns:
  - word_dict (dict): A dictionary where keys are unique words found in the emails, and values
    are dictionaries containing the frequency of each word for spam (1) and not spam (0) emails.
  """

  word_dict = {}

  num_emails = len(X)

  for i in range(num_emails):
    email = X[i]
    email_class = Y[i]
    email = set(email)

    for word in email:
      if word not in word_dict.keys():
        word_dict[word] = {"spam": 1, "ham":1}
      if email_class == 0:
        word_dict[word]["ham"] += 1
      if email_class == 1:
        word_dict[word]["spam"] += 1

  return word_dict

In [24]:
test_output = get_word_frequency([['like','going','river'], ['love', 'deep', 'river'], ['hate','river']], [1,0,0])
print(test_output)

{'going': {'spam': 2, 'ham': 1}, 'like': {'spam': 2, 'ham': 1}, 'river': {'spam': 2, 'ham': 3}, 'deep': {'spam': 1, 'ham': 2}, 'love': {'spam': 1, 'ham': 2}, 'hate': {'spam': 1, 'ham': 2}}


In [25]:
# This will build the word_frequency dictionary using the training set.
word_frequency = get_word_frequency(X_train,Y_train)

In [27]:
# To count the spam and ham emails, you may just sum the respective 1 and 0 values in the training dataset, since the convention is spam = 1 and ham = 0.
class_frequency = {'ham': sum(Y_train == 0), 'spam': sum(Y_train == 1)}
print(class_frequency)


{'ham': np.int64(3468), 'spam': np.int64(1114)}


In [28]:
# The idea is to compute  (amount of spam emails)/(total emails).
# Since an email is either spam or ham, total emails = (amount of ham emails) + (amount of spam emails).
proportion_spam = class_frequency['spam']/(class_frequency['ham'] + class_frequency['spam'])
print(f"The proportion of spam emails in training is: {proportion_spam:.4f}")

The proportion of spam emails in training is: 0.2431


## Next lets implement:

$$P(\text{word}_i \mid \text{class}) = \frac{\text{# emails in the class (either spam or ham) containing } \text{word}_i}{\text{# emails in the given class (spam or ham)}}$$

In [32]:
def p_word_given_class(word: str, cls:str, word_frequency:dict[dict], class_frequency:dict[int])->float:
    """
    Calculate the conditional probability of a given word occurring in a specific class.

    Parameters:
    - word (str): The target word for which the probability is calculated.
    - cls (str): The class for which the probability is calculated, it may be 'spam' or 'ham'
    - word_frequency (dict): The dictionary containing the words frequency.
    - class_frequency (dict): The dictionary containing the class frequency.

    Returns:
    - float: The conditional probability of the given word occurring in the specified class.
    """

    # Get the amount of times the word appears with the given class (class is stores in spam variable)
    amount_word_and_class = word_frequency[word][cls]
    p_word_given_class = amount_word_and_class/class_frequency[cls]

    return p_word_given_class

In [33]:
print(f"P(lottery | spam) = {p_word_given_class('lottery', cls = 'spam', word_frequency = word_frequency, class_frequency = class_frequency)}")
print(f"P(lottery | ham) = {p_word_given_class('lottery', cls = 'ham', word_frequency = word_frequency, class_frequency = class_frequency)}")
print(f"P(schedule | spam) = {p_word_given_class('schedule', cls = 'spam', word_frequency = word_frequency, class_frequency = class_frequency)}")
print(f"P(schedule | ham) = {p_word_given_class('schedule', cls = 'ham', word_frequency = word_frequency, class_frequency = class_frequency)}")

P(lottery | spam) = 0.00807899461400359
P(lottery | ham) = 0.0002883506343713956
P(schedule | spam) = 0.008976660682226212
P(schedule | ham) = 0.10294117647058823


## Now applying the naive assumption that:

$$P(\text{email} \mid \text{class}) = P(\text{word}_1 \mid \text{class}) \cdot P(\text{word}_2 \mid \text{class}) \cdots P(\text{word}_n \mid \text{class})$$

Recall that we are solving for $$P(\text{email} \mid \text{class})$$ in the bayes formula below:


$$ P(\text{spam} \mid \text{email}) = \frac{P(\text{email} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{email})} $$

In [36]:
def p_email_given_class(treated_email: list, cls: str, word_frequency:dict[dict], class_frequency:dict[int]):
    """
    Calculate the probability of an email being of a certain class (e.g., spam or ham) based on treated email content.

    Parameters:
    - treated_email (list): A list of treated words in the email.
    - cls (str): The class label for the email. It can be either 'spam' or 'ham'
    - word_frequency (dict): The dictionary containing the words frequency.
    - class_frequency (dict): The dictionary containing the class frequency.

    Returns:
    - float: The probability of the given email belonging to the specified class.
    """
    # prob starts at 1 because it will be updated by multiplying it with the current P(word | class) in every iteration
    prob = 1

    for word in treated_email:
      if word in word_frequency.keys():
        prob *= word_frequency[word][cls]/class_frequency[cls]

    return prob

In [37]:
example_email = "Click here to win a lottery ticket and claim your prize!"
treated_email = preprocess_text(example_email)
prob_spam = p_email_given_class(treated_email, cls = 'spam', word_frequency = word_frequency, class_frequency = class_frequency)
prob_ham = p_email_given_class(treated_email, cls = 'ham', word_frequency = word_frequency, class_frequency = class_frequency)
print(f"Email: {example_email}\nEmail after preprocessing: {treated_email}\nP(email | spam) = {prob_spam}\nP(email | ham) = {prob_ham}")

Email: Click here to win a lottery ticket and claim your prize!
Email after preprocessing: ['click' 'win' 'lottery' 'ticket' 'claim' 'prize']
P(email | spam) = 5.3884806600117164e-11
P(email | ham) = 1.2428344868918976e-15


## Putting it all together

We compute the probability an email is either spam or ham:

- $ P(\text{spam}) \cdot P(\text{email} \mid \text{spam}) $

- $ P(\text{ham}) \cdot P(\text{email} \mid \text{ham})$

The one with the greatest value will be the class your algorithm assigns to that email. Note that the function below includes a parameter that tells the function to return both probabilities rather than the class that was chosen.

In [41]:
def naive_bayes(treated_email: list, word_frequency: dict[dict], class_frequency:dict[int], return_likelihood:bool = False)-> int:
    """
    Naive Bayes classifier for spam detection.

    This function calculates the probability of an email being spam (1) or ham (0)
    based on the Naive Bayes algorithm. It uses the conditional probabilities of the
    treated_email given spam and ham, as well as the prior probabilities of spam and ham
    classes. The final decision is made by comparing the calculated probabilities.

    Parameters:
    - treated_email (list): A preprocessed representation of the input email.
    - word_frequency (dict): The dictionary containing the words frequency.
    - class_frequency (dict): The dictionary containing the class frequency.
        - return_likelihood (bool): If true, it returns the likelihood of both spam and ham.

    Returns:
    If return_likelihood = False:
        - int: 1 if the email is classified as spam, 0 if classified as ham.
    If return_likelihood = True:
        - tuple: A tuple with the format (spam_likelihood, ham_likelihood)
    """

    p_email_given_spam = p_email_given_class(treated_email, 'spam',word_frequency, class_frequency)
    p_email_given_ham = p_email_given_class(treated_email, 'ham',word_frequency, class_frequency)

    # Fix: Calculate priors using the total number of training emails in class_frequency
    total_train_emails = class_frequency['spam'] + class_frequency['ham']
    p_spam = class_frequency['spam'] / total_train_emails
    p_ham = 1 - p_spam

    spam_likelihood = p_spam * p_email_given_spam
    ham_likelihood = p_ham * p_email_given_ham

    # In case of passing return_likelihood = True, then return the desired tuple
    if return_likelihood == True:
        return (spam_likelihood, ham_likelihood)

    # Compares both values and choose the class corresponding to the higher value
    elif spam_likelihood >= ham_likelihood:
        return 1
    else:
        return 0

In [42]:
example_email = "Click here to win a lottery ticket and claim your prize!"
treated_email = preprocess_text(example_email)

print(f"Email: {example_email}\nEmail after preprocessing: {treated_email}\nNaive Bayes predicts this email as: {naive_bayes(treated_email, word_frequency, class_frequency)}")

print("\n\n")
example_email = "Our meeting will happen in the main office. Please be there in time."
treated_email = preprocess_text(example_email)

print(f"Email: {example_email}\nEmail after preprocessing: {treated_email}\nNaive Bayes predicts this email as: {naive_bayes(treated_email, word_frequency, class_frequency)}")

Email: Click here to win a lottery ticket and claim your prize!
Email after preprocessing: ['click' 'win' 'lottery' 'ticket' 'claim' 'prize']
Naive Bayes predicts this email as: 1



Email: Our meeting will happen in the main office. Please be there in time.
Email after preprocessing: ['meeting' 'happen' 'main' 'office' 'please' 'time']
Naive Bayes predicts this email as: 0


# Model Performance

We measure performance by computing the accuracy by:
- Counting every spam email that the model correctly classifies as spam (**true positives**)
- Counting every ham email that the model correctly classifies as ham (**true negatives**)

Finally, to get a proportion, you divide the sum of the true positives and true negatives by the total number of observations.


In [43]:
def get_true_positives(Y_true: list[int], Y_pred:list[int])->int:
    """
    Calculate the number of true positive instances in binary classification.

    Parameters:
    - Y_true (list): List of true labels (0 or 1) for each instance.
    - Y_pred (list): List of predicted labels (0 or 1) for each instance.

    Returns:
    - int: Number of true positives, where true label and predicted label are both 1.
    """
    # Both Y_true and Y_pred must match in length.
    if len(Y_true) != len(Y_pred):
        return "Number of true labels and predict labels must match!"
    n = len(Y_true)
    true_positives = 0
    # Iterate over the number of elements in the list
    for i in range(n):
        # Get the true label for the considered email
        true_label_i = Y_true[i]
        # Get the predicted (model output) for the considered email
        predicted_label_i = Y_pred[i]
        # Increase the counter by 1 only if true_label_i = 1 and predicted_label_i = 1 (true positives)
        if true_label_i == 1 and predicted_label_i == 1:
            true_positives += 1
    return true_positives

In [44]:
def get_true_negatives(Y_true:list[int], Y_pred:list[int])->int:
    """
    Calculate the number of true negative instances in binary classification.

    Parameters:
    - Y_true (list): List of true labels (0 or 1) for each instance.
    - Y_pred (list): List of predicted labels (0 or 1) for each instance.

    Returns:
    - int: Number of true negatives, where true label and predicted label are both 0.
    """

    # Both Y_true and Y_pred must match in length.
    if len(Y_true) != len(Y_pred):
        return "Number of true labels and predict labels must match!"
    n = len(Y_true)
    true_negatives = 0
    # Iterate over the number of elements in the list
    for i in range(n):
        # Get the true label for the considered email
        true_label_i = Y_true[i]
        # Get the predicted (model output) for the considered email
        predicted_label_i = Y_pred[i]
        # Increase the counter by 1 only if true_label_i = 0 and predicted_label_i = 0 (true negatives)
        if true_label_i == 0 and predicted_label_i == 0:
            true_negatives += 1
    return true_negatives

In [45]:
# Let's get the predictions for the test set:

# Create an empty list to store the predictions
Y_pred = []


# Iterate over every email in the test set
for email in X_test:
    # Perform prediction
    prediction = naive_bayes(email, word_frequency, class_frequency)
    # Add it to the list
    Y_pred.append(prediction)

# Checking if both Y_pred and Y_test (these are the true labels) match in length:
print(f"Y_test and Y_pred matches in length? Answer: {len(Y_pred) == len(Y_test)}")

Y_test and Y_pred matches in length? Answer: True


In [46]:
# Get the number of true positives:
true_positives = get_true_positives(Y_test, Y_pred)

# Get the number of true negatives:
true_negatives = get_true_negatives(Y_test, Y_pred)

print(f"The number of true positives is: {true_positives}\nThe number of true negatives is: {true_negatives}")

# Compute the accuracy by summing true negatives with true positives and dividing it by the total number of elements in the dataset.
# Since both Y_pred and Y_test have the same length, it does not matter which one you use.
accuracy = (true_positives + true_negatives)/len(Y_test)

print(f"Accuracy is: {accuracy:.4f}")

The number of true positives is: 249
The number of true negatives is: 723
Accuracy is: 0.8482


# Underflow Problem
A hidden problem in the current model is impacting its performance. Given that we are multiplying the probability of each word in an email, our result is highly likely to be $0$ at some point if an email has many words

The challenge you encounter is termed an **underflow problem**, indicating that you are dealing with exceedingly small numbers beyond the computer's precision. In this case, the root cause is the **very large product** involved in Naive Bayes calculations. Fortunately, there is a solution to this issue.

We can resolve this using the log of the probabilities which preserves the properties and ordering probabilites and transforms products into sums. Therefore, we can compare the following quantities:

$$\log \left(P(\text{spam}) \cdot P(\text{email} \mid \text{spam}) \right)$$
$$\log \left(P (\text{ham}) \cdot P(\text{email} \mid \text{ham}) \right)$$

And choose the maximum value among these new quantities. Denoting the class as either spam or ham:

$$\log \left(P(\text{class}) \cdot P(\text{email} \mid \text{class}) \right) = \log \left(P(\text{class}) \right) + \log \left( P(\text{email} \mid \text{class}) \right)$$

And

$$\log \left( P(\text{email} \mid \text{class}) \right) = \log  \left(P(\text{word}_1 \mid \text{class}) \cdot P(\text{word}_2 \mid \text{class}) \cdots P(\text{word}_n \mid \text{class}) \right) = \log  \left(P(\text{word}_1 \mid \text{class}) \right) + \log \left(P(\text{word}_2 \mid \text{class})\right) + \cdots + \log \left( P(\text{word}_n \mid \text{class}) \right) $$

With this approach, you have transformed a large product into a large summation, a significantly more numerically stable operation. Now, you will improve our functions with this new technique! You need to adjust two functions:

- `p_email_given_class` - replace the probability word product by the sum of the logs
- `naive_bayes` - replace the product $P(\text{class}) \cdot P(\text{email} \mid \text{class})$ by its respective sum of log.

The new functions will be called `log_p_email_given_class` and `log_naive_bayes`.

In [51]:
def log_p_email_given_class(treated_email, cls, word_frequency, class_frequency):
    """
    Calculate the log probability of an email being of a certain class (e.g., spam or ham) based on treated email content.

    Parameters:
    - treated_email (list): A list of treated words in the email.
    - cls (str): The class label ('spam' or 'ham')


    Returns:
    - float: The log probability of the given email belonging to the specified class.
    """

    # prob starts at 0 because it will be updated by summing it with the current log(P(word | class)) in every iteration
    prob = 0

    for word in treated_email:
        # Only perform the computation for words that exist in the word frequency dictionary
        if word in word_frequency.keys():
            # Update the prob by summing it with log(P(word | class))
            prob += np.log(p_word_given_class(word, cls,word_frequency, class_frequency))

    return prob

In [48]:
def log_naive_bayes(treated_email, word_frequency, class_frequency, return_likelihood = False):
    """
    Naive Bayes classifier for spam detection, comparing the log probabilities instead of the actual probabilities.

    This function calculates the log probability of an email being spam (1) or ham (0)
    based on the Naive Bayes algorithm. It uses the conditional probabilities of the
    treated_email given spam and ham, as well as the prior probabilities of spam and ham
    classes. The final decision is made by comparing the calculated probabilities.

    Parameters:
    - treated_email (list): A preprocessed representation of the input email.
    - return_likelihood (bool): If true, it returns the log_likelihood of both spam and ham.

    Returns:
    - int: 1 if the email is classified as spam, 0 if classified as ham.
    """

    # Compute P(email | spam) with the new log function
    log_prob_email_given_spam = log_p_email_given_class(treated_email, cls = 'spam',word_frequency = word_frequency, class_frequency = class_frequency)

    # Compute P(email | ham) with the function you defined just above
    log_prob_email_given_ham = log_p_email_given_class(treated_email, cls = 'ham',word_frequency = word_frequency, class_frequency = class_frequency)

    # Compute P(spam) using the class_frequency dictionary and using the formula #spam emails / #total emails
    p_spam = class_frequency['spam']/(class_frequency['ham'] + class_frequency['spam'])

    # Compute P(ham) using the class_frequency dictionary and using the formula #ham emails / #total emails
    p_ham = class_frequency['ham']/(class_frequency['ham'] + class_frequency['spam'])

    # Compute the quantity log(P(spam)) + log(P(email | spam)), let's call it log_spam_likelihood
    log_spam_likelihood = np.log(p_spam) + log_prob_email_given_spam

    # Compute the quantity P(ham) * P(email | ham), let's call it ham_likelihood
    log_ham_likelihood = np.log(p_ham) + log_prob_email_given_ham

    # In case of passing return_likelihood = True, then return the desired tuple
    if return_likelihood == True:
        return (log_spam_likelihood, log_ham_likelihood)

    # Compares both values and choose the class corresponding to the higher value.
    # As the logarithm is an increasing function, the class with the higher value retains this property.
    if log_spam_likelihood >= log_ham_likelihood:
        return 1
    else:
        return 0

In [49]:
log_spam_likelihood, log_ham_likelihood = log_naive_bayes(treated_email,word_frequency = word_frequency, class_frequency = class_frequency,return_likelihood = True)
print(f"log_spam_likelihood: {log_spam_likelihood}\nlog_ham_likelihood: {log_ham_likelihood}")

log_spam_likelihood: -21.726186604479015
log_ham_likelihood: -13.320197381096177


In [50]:
# Let's get the predictions for the test set:

# Create an empty list to store the predictions
Y_pred = []


# Iterate over every email in the test set
for email in X_test:
    # Perform prediction
    prediction = log_naive_bayes(email,word_frequency = word_frequency, class_frequency = class_frequency)
    # Add it to the list
    Y_pred.append(prediction)

# Get the number of true positives:
true_positives = get_true_positives(Y_test, Y_pred)

# Get the number of true negatives:
true_negatives = get_true_negatives(Y_test, Y_pred)

print(f"The number of true positives is: {true_positives}\nThe number of true negatives is: {true_negatives}")

# Compute the accuracy by summing true negatives with true positives and dividing it by the total number of elements in the dataset.
# Since both Y_pred and Y_test have the same length, it does not matter which one you use.
accuracy = (true_positives + true_negatives)/len(Y_test)

print(f"The accuracy is: {accuracy:.4f}")

The number of true positives is: 249
The number of true negatives is: 888
The accuracy is: 0.9921
